In [1]:
import pandas as pd
from pathlib import Path
import sys
import spacy
import warnings

In [2]:
sys.path.append("../src")
from entities import EntityResolver

In [3]:
# Ignoramos warnings de spaCy para mantener la salida limpia
warnings.filterwarnings("ignore")

In [4]:
BASE_DIR = Path.cwd().parent
CSV_CHATS = BASE_DIR / "data" / "processed" / "chats_completos.csv"
EDGES_CSV_RAW = BASE_DIR / "data" / "processed" / "grafo_aristas.csv"
EDGES_CSV_CLEAN = BASE_DIR / "data" / "processed" / "grafo_aristas_resuelto.csv"

In [5]:
df_chats = pd.read_csv(CSV_CHATS)
nlp = spacy.load("es_core_news_md", disable=["parser", "attribute_ruler", "lemmatizer"])

In [6]:
def construir_grafo_avanzado(df):
    aristas = []
    
    # 1. Limpieza y preparación secuencial
    df = df.dropna(subset=['Emisor', 'Mensaje']).copy()
    df['Emisor_Limpio'] = df['Emisor'].str.upper().str.strip()
    df = df.reset_index(drop=True)
    
    print("Analizando dinámicas conversacionales y menciones...")
    
    for i in range(len(df)):
        emisor_actual = df.loc[i, 'Emisor_Limpio']
        mensaje = str(df.loc[i, 'Mensaje'])
        pagina = df.loc[i, 'Pagina_PDF']
        
        # --- LÓGICA 1: ARISTAS DIRECTAS (Comunicación Secuencial) ---
        # Buscamos quién responde. Miramos los siguientes 3 mensajes como ventana de conversación.
        for j in range(i + 1, min(i + 4, len(df))):
            emisor_siguiente = df.loc[j, 'Emisor_Limpio']
            
            # Si el emisor cambia, inferimos que es una respuesta/conversación directa
            if emisor_actual != emisor_siguiente:
                aristas.append({
                    "Origen": emisor_actual,
                    "Destino": emisor_siguiente,
                    "Tipo_Relacion": "COMUNICA_CON",
                    "Pagina": pagina
                })
                break # Solo conectamos con el receptor inmediato, luego saltamos al siguiente
                
        # --- LÓGICA 2: ARISTAS INDIRECTAS (Menciones / NLP) ---
        # Extraemos de quién están hablando en el mensaje
        doc = nlp(mensaje)
        for ent in doc.ents:
            # Filtramos entidades de tipo Persona largas para evitar ruido (ej. iniciales)
            if ent.label_ == "PER" and len(ent.text) > 3:
                mencionado = ent.text.upper().strip()
                
                # Evitamos auto-menciones
                if emisor_actual != mencionado:
                    aristas.append({
                        "Origen": emisor_actual,
                        "Destino": mencionado,
                        "Tipo_Relacion": "MENCIONA_A",
                        "Pagina": pagina
                    })
                    
    return pd.DataFrame(aristas)

In [7]:
if not EDGES_CSV_RAW.exists():
    df_aristas_raw = construir_grafo_avanzado(df_chats)
    
    # Agrupamos para calcular el "Peso" de cada relación (cuántas interacciones o menciones hay)
    df_grafo = df_aristas_raw.groupby(['Origen', 'Destino', 'Tipo_Relacion']).size().reset_index(name='Peso')
    df_grafo = df_grafo.sort_values(by='Peso', ascending=False)
    
    df_grafo.to_csv(EDGES_CSV_RAW, index=False)
    print("Grafo heterogéneo extraído y guardado con éxito.")
else:
    print("Cargando aristas previamente extraídas...")
    df_grafo = pd.read_csv(EDGES_CSV_RAW)

Analizando dinámicas conversacionales y menciones...
Grafo heterogéneo extraído y guardado con éxito.


In [8]:
# 1. Cargamos el grafo crudo que sacamos en el paso anterior
df_grafo_crudo = pd.read_csv(EDGES_CSV_RAW)

# 2. Instanciamos nuestra clase (podemos inyectarle nuevos alias si descubrimos más)
resolver = EntityResolver()

# 3. Consolidamos la red matemática
df_grafo_limpio = resolver.consolidar_grafo(
    df_grafo=df_grafo_crudo, 
    columnas_entidades=["Origen", "Destino"]
)

In [9]:
# Guardamos el grafo dorado listo para Machine Learning
df_grafo_limpio.to_csv(EDGES_CSV_CLEAN, index=False)

# Validación
print(f"Nodos consolidados. Aristas originales: {len(df_grafo_crudo)} | Aristas limpias: {len(df_grafo_limpio)}")

Nodos consolidados. Aristas originales: 570 | Aristas limpias: 519


In [10]:
print(f"\nTotal de relaciones consolidadas: {len(df_grafo_limpio)}")
print("\nTop 5 Relaciones de Comunicación Directa:")
display(df_grafo_limpio[df_grafo_limpio['Tipo_Relacion'] == 'COMUNICA_CON'].head(5))

print("\nTop 5 Relaciones de Influencia (Menciones):")
display(df_grafo_limpio[df_grafo_limpio['Tipo_Relacion'] == 'MENCIONA_A'].head(5))


Total de relaciones consolidadas: 519

Top 5 Relaciones de Comunicación Directa:


,Origen,Destino,Tipo_Relacion,Peso
0,MIGUEL PALOMERO,RODOLFO REYES,COMUNICA_CON,94
1,RODOLFO REYES,JULIO MARTINEZ SOLA,COMUNICA_CON,86
2,ROBERTO ROSELLI,RODOLFO REYES,COMUNICA_CON,77
3,JULIO MARTINEZ SOLA,RODOLFO REYES,COMUNICA_CON,76
4,RODOLFO REYES,MIGUEL PALOMERO,COMUNICA_CON,68



Top 5 Relaciones de Influencia (Menciones):


,Origen,Destino,Tipo_Relacion,Peso
6,RODOLFO REYES,JULIO MARTINEZ SOLA,MENCIONA_A,30
7,JULIO MARTINEZ SOLA,RODOLFO REYES,MENCIONA_A,29
8,ROBERTO ROSELLI,JULIO MARTINEZ MARTINEZ,MENCIONA_A,23
9,ROBERTO ROSELLI,JULIO MARTINEZ SOLA,MENCIONA_A,22
10,ROBERTO ROSELLI,RODOLFO REYES,MENCIONA_A,22
